### **Vision-Language Medical Foundation Models**

#### **1.2. Contrastive text-image pre-training**
---

**The most popular paradigm for vision-language pre-training was introduced in CLIP [1]**. Given a dataset with **paired images and text descriptions**, a **vision and a text encoder** are pre-trained to **produce a joint embedding space** in which paired data produce similar representations, which are pushed away from unpaired samples.

Note that pre-training is time-consuming, and you require a large dataset with image-text pairs, which are scarce in medical imaging. In this notebook, we will simply do a toy example to compute the CLIP **contrastive language-image pre-training loss**.



In [ ]:
# General imports 
import warnings
warnings.filterwarnings('ignore')

import copy
import shutil
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

# Device for training/inference
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Available device: " + device)


In [ ]:
# Dataset setup: copy SICAPv2 from shared project storage to local storage if needed.
# This makes the notebook portable across student accounts.

SHARED_DATA_ROOT = Path.home() / "projects" / "def-sponsor00" / "SICAPv2"
LOCAL_DATA_ROOT = Path.home() / "local_data" / "datasets" / "SICAPv2"

if not LOCAL_DATA_ROOT.exists():
    LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print("Copying SICAPv2 from shared project folder to local_data...")
    shutil.copytree(SHARED_DATA_ROOT, LOCAL_DATA_ROOT)
else:
    print("SICAPv2 already exists in local_data.")

DATA_ROOT = LOCAL_DATA_ROOT
IMAGE_DIR = DATA_ROOT / "images"

assert IMAGE_DIR.exists(), f"Image directory not found: {IMAGE_DIR}"
print("Dataset ready at:", DATA_ROOT)


#### **VLM model wrapper**

In [ ]:
# Load model and pre-processing tools from HuggingFace
from transformers import AutoProcessor, AutoModel

# In the Transformers library, models and versions are stored by an ID defining the model name.
# For the PLIP model, this ID is "vinid/plip".
processor = AutoProcessor.from_pretrained("vinid/plip")  # pre-processing image and text
processor.image_processor.do_center_crop = False

# Model with pre-trained weights.
# We move it to device so that the model and input tensors are on the same device.
plip = AutoModel.from_pretrained("vinid/plip").eval().to(device)


In [ ]:
# Again, we will use our PLIP Wrapper for easy interaction.
class PLIPWrapper(torch.nn.Module):
    def __init__(self, encoder, proj_layer):
        super().__init__()

        self.encoder = encoder         # One-modality encoder from VLM.
        self.proj_layer = proj_layer   # Projection layer into the joint embedding space.

    def forward(self, inputs):
        # Forward input through encoder.
        features = self.encoder(**inputs).pooler_output  # Global feature for the image/text.

        # Project features into the shared embedding space.
        projected_features = self.proj_layer(features)

        # L2-normalize features.
        projected_features = projected_features / projected_features.norm(dim=-1, keepdim=True)

        return projected_features

# Create model wrappers for vision and text encoders.
vision_encoder = PLIPWrapper(plip.vision_model, plip.visual_projection).to(device).eval()
text_encoder = PLIPWrapper(plip.text_model, plip.text_projection).to(device).eval()


#### **Contrastive pre-training**

Now, we are going to compute the CLIP pre-training loss. We will do an example with few images, and naive text descriptions

In [ ]:
# First, let's load a few examples, one for each category.
nc = Image.open(IMAGE_DIR / "16B0028148_Block_Region_8_2_14_xini_27858_yini_55555.jpg").convert("RGB")
g3 = Image.open(IMAGE_DIR / "18B0006169B_Block_Region_6_3_7_xini_33958_yini_90365.jpg").convert("RGB")
g4 = Image.open(IMAGE_DIR / "17B0032153_Block_Region_10_13_17_xini_23105_yini_15687.jpg").convert("RGB")
g5 = Image.open(IMAGE_DIR / "16B0008067_Block_Region_0_6_2_xini_10859_yini_103113.jpg").convert("RGB")

# Pre-process images and concatenate them as a batched input.
image_inputs = processor.image_processor([nc, g3, g4, g5], return_tensors="pt")
image_inputs = {k: v.to(device) for k, v in image_inputs.items()}
print("Image batch shape:", image_inputs["pixel_values"].shape)

# Naive text descriptions for the toy example. These are treated as paired with each image.
prompt = [
    "Healthy prostate tissue",
    "Gleason grade 3",
    "Gleason grade 4",
    "Gleason grade 5",
]

# Pre-process text data.
text_inputs = processor.tokenizer(
    prompt,
    max_length=77,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

# Forward representations into the common image-text embedding space.
with torch.no_grad():
    image_features = vision_encoder(image_inputs)
    text_features = text_encoder(text_inputs)

# Compute similarity matrix: image_features (B x D) @ text_features.T (D x B) -> (B x B).
# Each row gives the similarity of one image to all text descriptions.
# Ideally, the diagonal is high because image i should match text i.
with torch.no_grad():
    sim = image_features @ text_features.t() * plip.logit_scale.exp()

print("Predicted similarity matrix")
print(sim.detach().cpu())
print(" ")

# One-to-one target: image i is paired with text i.
target = torch.arange(text_features.shape[0], device=device)

# Image-to-text loss:
# Softmax over rows. For each image, the correct class is its paired text.
logits_per_image = sim
print("I2T Softmax:")
with torch.no_grad():
    print(str(torch.softmax(logits_per_image, dim=-1).detach().cpu().numpy().round(2)))
    print(" ")

i2t_loss = torch.nn.functional.cross_entropy(logits_per_image, target)

# Text-to-image loss:
# Softmax over columns, implemented by transposing the similarity matrix.
logits_per_text = logits_per_image.t()
print("T2I Softmax:")
with torch.no_grad():
    print(str(torch.softmax(logits_per_text, dim=-1).detach().cpu().numpy().round(2)))
    print(" ")

t2i_loss = torch.nn.functional.cross_entropy(logits_per_text, target)

# Overall CLIP loss.
clip_loss = (i2t_loss + t2i_loss) / 2
print("CLIP Loss:", clip_loss.item())

# This loss is computed on mini-batches and backpropagated through both encoders during pre-training.
# During training, the joint embedding space learns to align paired concepts across modalities.
# The temperature scaling parameter is also optimized during pre-training.


## **References**


[1] Radford, A., Kim, J.W., Hallacy, C., Ramesh, A., Goh, G., Agarwal, S., Sastry, G., Askell, A., Mishkin, P., Clark, J., Krueger, G., & Sutskever, I. (2021). Learning Transferable Visual Models From Natural Language Supervision. International Conference on Machine Learning. 

---